# Competitive Analysis


---

## 📦 Step 1: Import Libraries


In [1]:
import json
import os
import re
from pathlib import Path
from collections import Counter

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

PLOTLY_TEMPLATE = "simple_white"

In [2]:
pd.set_option("display.max_columns", 50)

---

## 📂 Step 2: Load & Parse JSON Files

Each file in `product-results/` is a detailed product page for one ASIN.  
We navigate the nested structure: `data["product"]["price"]["currentPrice"]`


In [7]:
PRODUCT_DIR = Path("product-results")


def extract_brand(title: str) -> str:
    # Extract the first word as the brand, or return "Unknown" if title is empty
    # Note that this is a very naive approach and will not be accurate for all products
    # There are more sophisticated methods (e.g. using a list of known brands, or NLP techniques)
    # For the purpose of this case study, we will manually clean the brand column later if needed
    return title.split()[0] if title else "Unknown"


def parse_product(filepath: Path) -> dict | None:
    """Load one product JSON and return a flat dict of key fields."""
    with open(filepath, "r", encoding="utf-8") as f:
        data = json.load(f)

    p = data.get("product", {})
    if not p:
        return None

    # Price
    price_obj = p.get("price", {})
    current_px = price_obj.get("currentPrice")
    before_px = price_obj.get("beforePrice")
    discount = price_obj.get("discount", "").replace("%", "")

    # Reviews
    rev_obj = p.get("reviewsInfo", {})
    rating = rev_obj.get("rating")
    total_rev = rev_obj.get("totalReviews")
    stars = rev_obj.get("starRates", {})
    five_star_pct = stars.get("fiveStars", "0%").replace("%", "")
    four_star_pct = stars.get("fourStars", "0%").replace("%", "")
    three_star_pct = stars.get("threeStars", "0%").replace("%", "")
    two_star_pct = stars.get("twoStars", "0%").replace("%", "")
    one_star_pct = stars.get("oneStars", "0%").replace("%", "")

    # Badges
    badges = p.get("badges", {})
    best_seller = badges.get("bestSeller", False)
    amz_choice = badges.get("amazonChoice", False)
    amz_exclusive = badges.get("amazonExclusive", False)
    amz_prime = badges.get("amazonPrime", False)
    limited_time_deal = badges.get("limitedTimeDeal", False)

    # Feature bullets (keywords)
    bullets = " ".join(p.get("featureBullets", []))
    description = p.get("description", "")

    # Bought in past month
    bought_raw = p.get("boughtInPastMonth", "")
    bought_num = None
    if bought_raw:
        bought_raw_clean = (
            bought_raw.replace("+", "").replace(",", "").replace("K", "000")
        )
        try:
            bought_num = int(bought_raw_clean)
        except ValueError:
            pass

    title = p.get("title", "")
    brand = extract_brand(title)

    return {
        "asin": p.get("asin"),
        "title": title,
        "amazon_url": p.get("url"),
        "brand": "",
        "is_makeup_product": "",
        "price": current_px,
        "before_price": before_px,
        "discount": float(discount) / 100 if discount else None,
        "rating": rating,
        "total_reviews": total_rev,
        "five_star_pct": float(five_star_pct) / 100 if five_star_pct else None,
        "four_star_pct": float(four_star_pct) / 100 if four_star_pct else None,
        "three_star_pct": float(three_star_pct) / 100 if three_star_pct else None,
        "two_star_pct": float(two_star_pct) / 100 if two_star_pct else None,
        "one_star_pct": float(one_star_pct) / 100 if one_star_pct else None,
        "seller": p.get("seller"),
        "is_best_seller": best_seller,
        "is_amazon_choice": amz_choice,
        "is_amazon_exclusive": amz_exclusive,
        "is_amazon_prime": amz_prime,
        "is_limited_time_deal": limited_time_deal,
        "bought_past_month": bought_num,
        "bullets": bullets,
        "description": description,
        "is_available": p.get("isAvailable", False),
    }

In [8]:
records = []

for fp in PRODUCT_DIR.glob("*.json"):
    row = parse_product(fp)
    if row:
        records.append(row)

df = pd.DataFrame(records)
print(f"Loaded {len(df)} products from {PRODUCT_DIR}/")

Loaded 227 products from product-results/


In [5]:
df

,asin,title,amazon_url,brand,price,before_price,discount,rating,total_reviews,five_star_pct,four_star_pct,three_star_pct,two_star_pct,one_star_pct,seller,is_best_seller,is_amazon_choice,is_amazon_exclusive,is_amazon_prime,is_limited_time_deal,bought_past_month,bullets,description,is_available
0,0310441870,"NIV, Foundation Study Bible, Leathersoft, Brow...",https://www.amazon.com/dp/0310441870,,16.47,39.99,-0.59,4.8,2981.0,0.90,0.07,0.02,0.00,0.01,Amazon.com,False,False,False,True,False,NaN,,The key features of a study Bible in a conveni...,True
1,1638380112,SCP Foundation Artbook | Paperback Edition | B...,https://www.amazon.com/dp/1638380112,,24.45,29.90,-0.18,4.8,76.0,0.91,0.07,0.00,0.00,0.02,Aloha Comics,False,False,False,False,False,NaN,,SCP Foundation Artbook Series by ParaBooks is ...,True
2,B000IJLQOW,"Revlon Liquid Foundation, ColorStay Face Makeu...",https://www.amazon.com/dp/B000IJLQOW,,12.48,12.48,-0.05,4.5,17834.0,0.76,0.13,0.06,0.02,0.03,Amazon.com,False,False,False,True,False,1000.0,"LONG-WEARING: Oil-free, fragrance-free foundat...",Product Description Provides medium to buildab...,True
3,B000QW6SXK,COVERGIRL Advanced Radiance Age Defying Founda...,https://www.amazon.com/dp/B000QW6SXK,,8.98,8.98,-0.05,4.5,12728.0,0.75,0.14,0.06,0.02,0.03,Amazon.com,False,True,False,True,False,500.0,Instantly restore your skin's youthful appeara...,This smooth liquid foundation with an exclusiv...,True
4,B000R8YZQA,Glo Skin Beauty Pressed Base Powder Foundation...,https://www.amazon.com/dp/B000R8YZQA,,52.00,52.00,-0.10,4.5,2767.0,0.77,0.10,0.05,0.02,0.06,GLO SKIN BEAUTY,False,False,False,True,False,300.0,LONG-WEAR POWDER FOUNDATION – This pressed pow...,,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
222,B0GQT12YQ8,"Color Changing Foundation for Mature Skin, 202...",https://www.amazon.com/dp/B0GQT12YQ8,,15.95,NaN,NaN,2.4,5.0,0.28,0.00,0.00,0.27,0.45,KAMOJI(7-15 Days Fast Delivery),False,False,False,False,False,NaN,"The weightless consistency creates a radiant, ...",Specification: Texture: Magical Color Changing...,True
223,B0GR86CS8T,"Hush Foundation for Older Women,Warm，Perfect f...",https://www.amazon.com/dp/B0GR86CS8T,,17.99,NaN,NaN,4.4,5.0,0.69,0.00,0.31,0.00,0.00,Nanyangyanyingbaihuoyouxiangongsi,False,False,False,True,False,NaN,Formulated for Mature Skin – This cushion foun...,"Hush Foundation for Older Women,Warm",True
224,B0GRGKJMY3,"Color Changing Foundation Stick, Dual-Ended Ko...",https://www.amazon.com/dp/B0GRGKJMY3,,24.99,NaN,NaN,2.0,3.0,0.26,0.00,0.00,0.00,0.74,Nuvie Official,False,False,False,True,False,NaN,Smart Color-Matching Formula：This innovative K...,,True
225,B0GTPFZSQD,"4-in-1 Color Changing Foundation, Long-Lasting...",https://www.amazon.com/dp/B0GTPFZSQD,,17.99,NaN,NaN,4.6,31.0,0.85,0.06,0.00,0.00,0.09,Daily Care Hub,False,False,False,True,False,NaN,✨【Perfect Skin Tone Matching】This color changi...,,True


In [9]:
df.to_csv("foundation_products.csv", index=False)

---

## 🧹 Step 3: Data Cleaning & Feature Engineering


First, check for outliers in the price column.


In [87]:
df.sort_values("price", ascending=False)[["title", "price"]].head(10)

,title,price
5,"COVERGIRL Smoothers Lightweight BB Cream, Fair...",45152.00
124,NYX PROFESSIONAL MAKEUP Bare With Me Blur Skin...,19575.50
10,Mineral Fusion Pressed Powder Foundation - Mat...,593.85
82,"Amazon Basics Smart Box Spring Bed Base, 5-Inc...",86.00
185,"No Makeup Makeup Foundation + Brush Duo, Cream...",80.00
144,"EMODA 5 Inch Box Spring Queen Size Bed Base, H...",69.99
214,Hush Glow™ Foundation Cushion SPF 30 – Skincar...,59.93
93,Estée Lauder Futurist Hydra Rescue Moisturizin...,55.00
164,"No Makeup Makeup Foundation, Weightless Cream-...",55.00
95,Lancôme Renergie Lift Makeup Foundation - Ligh...,53.55


There are two products with prices over $10k, which are likely outliers or errors. We will only compare products in the $0-$200 range, which is more relevant for our analysis of affordable foundations.


In [88]:
df = df[df["price"] <= 200].copy()

In [89]:
# Drop rows without a price (non-makeup products that slipped in)
df = df[df["price"].notna()].copy()

# Price tier buckets
bins = [0, 10, 20, 30, 50, 200]
labels = ["Under $10", "$10–$20", "$20–$30", "$30–$50", "$50+"]
df["price_tier"] = pd.cut(df["price"], bins=bins, labels=labels)

# # Keyword flags for positioning analysis
# keyword_map = {
#     "full_coverage": r"full.?coverage",
#     "medium_coverage": r"medium.?coverage",
#     "lightweight": r"lightweight|light.weight|light weight",
#     "hydrating": r"hydrat",
#     "matte": r"matte",
#     "dewy": r"dewy|dew",
#     "natural_finish": r"natural.?finish",
#     "spf": r"spf",
#     "long_wear": r"long.?wear|24.?hour|24hr",
#     "skin_tint": r"skin.?tint",
#     "foundation_stick": "stick",
#     "buildable": r"buildable",
# }

# combined_text = (
#     df["title"] + " " + df["bullets"] + " " + df["description"]
# ).str.lower()

# for kw, pattern in keyword_map.items():
#     df[kw] = combined_text.str.contains(pattern, na=False, regex=True).astype(int)

# print("✅ Cleaning done. Shape:", df.shape)
# print("\nNull counts:")
# print(df[["price", "rating", "total_reviews"]].isna().sum())

---

## 📊 Step 4: Exploratory Analysis

### 4.1 Dataset Summary


In [90]:
df[["price", "rating", "total_reviews", "bought_past_month"]].describe().round(2)

,price,rating,total_reviews,bought_past_month
count,213.00,211.00,211.00,198.00
mean,22.12,4.29,8250.34,1559.60
std,15.19,0.35,18118.09,2668.58
min,4.92,2.00,3.00,200.00
25%,9.99,4.10,492.50,400.00
50%,16.02,4.40,2279.00,700.00
75%,32.00,4.50,8359.50,1750.00
max,86.00,4.80,154234.00,20000.00


### 4.2 Price Distribution — What Price Range Dominates?


In [91]:
fig = px.histogram(
    df.dropna(subset=["price"]),
    x="price",
    title="💰 Price Distribution of Best-Selling Foundation Products",
    labels={"price": "Price (USD)", "count": "# Products"},
    template=PLOTLY_TEMPLATE,
)
fig.show()

### 4.3 Price Tier Share


In [92]:
tier_counts = df["price_tier"].value_counts().reset_index()
tier_counts.columns = ["price_tier", "count"]

fig = px.pie(
    tier_counts,
    names="price_tier",
    values="count",
    title="Price Tier Distribution of Best-Selling Foundations",
    color_discrete_sequence=px.colors.sequential.Oranges,
    template=PLOTLY_TEMPLATE,
    hole=0.35,
)
fig.update_traces(textposition="outside", textinfo="percent+label")
fig.show()

### 4.4 Rating Distribution


In [93]:
fig = px.histogram(
    df.dropna(subset=["rating"]),
    x="rating",
    nbins=20,
    title="Rating Distribution Across Best-Selling Foundations",
    labels={"rating": "Star Rating", "count": "# Products"},
    color_discrete_sequence=["green"],
    template=PLOTLY_TEMPLATE,
)
fig.show()

### 4.5 Do Higher-Rated Products Also Have More Reviews?


In [94]:
plot_df = df.dropna(subset=["rating", "total_reviews", "price"]).copy()

fig = px.scatter(
    plot_df,
    x="rating",
    y="total_reviews",
    size="price",
    color="brand",
    hover_name="title",
    hover_data={"price": True, "rating": True, "total_reviews": True},
    title="Rating vs. Total Reviews (bubble size = price)",
    labels={
        "rating": "Avg Star Rating",
        "total_reviews": "Total Reviews",
        "price": "Price ($)",
    },
    template=PLOTLY_TEMPLATE,
    log_y=True,
)
fig.update_layout(legend_title_text="Brand")
fig.show()

# Pearson correlation
corr = plot_df[["rating", "total_reviews"]].corr().iloc[0, 1]
print(f"Pearson correlation (rating vs. log reviews): {corr:.3f}")

Pearson correlation (rating vs. log reviews): 0.186


### 4.6 Brand Dominance — Who Owns the Category?


In [95]:
brand_df = (
    df.groupby("brand", as_index=False)
    .agg(
        product_count=("asin", "count"),
        avg_price=("price", "mean"),
        avg_rating=("rating", "mean"),
        total_reviews=("total_reviews", "sum"),
    )
    .sort_values("product_count", ascending=False)
    .head(15)
)

fig = px.bar(
    brand_df,
    x="product_count",
    y="brand",
    orientation="h",
    color="avg_rating",
    color_continuous_scale="RdYlGn",
    title="🏷️ Top Brands by # of Best-Selling SKUs (color = avg rating)",
    labels={"product_count": "# of SKUs", "brand": "Brand", "avg_rating": "Avg Rating"},
    template=PLOTLY_TEMPLATE,
    text="product_count",
)
fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.show()

### 4.7 Price vs. Avg Rating by Brand


In [96]:
fig = px.scatter(
    brand_df,
    x="avg_price",
    y="avg_rating",
    size="total_reviews",
    text="brand",
    title="💵 Avg Price vs. Avg Rating by Brand (bubble size = total reviews)",
    labels={
        "avg_price": "Avg Price ($)",
        "avg_rating": "Avg Rating",
        "total_reviews": "Total Reviews",
    },
    template=PLOTLY_TEMPLATE,
    color="brand",
)
fig.update_traces(textposition="top center")
fig.update_layout(
    xaxis_tickprefix="$",
    yaxis_range=[4.0, 5.0],
    showlegend=False,
)
fig.show()

### 4.8 Product Positioning — Keyword Frequency

> Which product attributes are most commonly called out in titles, bullets, and descriptions?


In [97]:
# kw_cols = list(keyword_map.keys())
# kw_counts = df[kw_cols].sum().reset_index()
# kw_counts.columns = ["keyword", "count"]
# kw_counts = kw_counts.sort_values("count", ascending=True)

# # Friendlier labels
# label_map = {
#     "full_coverage": "Full Coverage",
#     "medium_coverage": "Medium Coverage",
#     "lightweight": "Lightweight",
#     "hydrating": "Hydrating",
#     "matte": "Matte Finish",
#     "dewy": "Dewy Finish",
#     "natural_finish": "Natural Finish",
#     "spf": "SPF / Sun Protection",
#     "long_wear": "Long Wear / 24H",
#     "skin_tint": "Skin Tint",
#     "foundation_stick": "Stick Format",
#     "buildable": "Buildable Coverage",
# }
# kw_counts["label"] = kw_counts["keyword"].map(label_map)

# fig = px.bar(
#     kw_counts,
#     x="count",
#     y="label",
#     orientation="h",
#     title="🔑 Product Positioning Keywords — How Foundations Are Marketed",
#     labels={"count": "# of Products Mentioning Keyword", "label": ""},
#     color="count",
#     color_continuous_scale="Reds",
#     template=PLOTLY_TEMPLATE,
#     text="count",
# )
# fig.update_layout(coloraxis_showscale=False)
# fig.show()

### 4.9 Positioning by Price Tier


In [98]:
kw_cols_top = [
    "full_coverage",
    "lightweight",
    "hydrating",
    "matte",
    "natural_finish",
    "long_wear",
    "spf",
]

tier_kw = (
    df.dropna(subset=["price_tier"])
    .groupby("price_tier", observed=True)[kw_cols_top]
    .mean()
    .round(2)
    .reset_index()
)

tier_kw_melted = tier_kw.melt(
    id_vars="price_tier", var_name="keyword", value_name="share"
)
tier_kw_melted["keyword"] = tier_kw_melted["keyword"].map(
    {
        "full_coverage": "Full Coverage",
        "lightweight": "Lightweight",
        "hydrating": "Hydrating",
        "matte": "Matte",
        "natural_finish": "Natural Finish",
        "long_wear": "Long Wear / 24H",
        "spf": "SPF",
    }
)

fig = px.bar(
    tier_kw_melted,
    x="price_tier",
    y="share",
    color="keyword",
    barmode="group",
    title="🎯 Product Positioning Attributes by Price Tier",
    labels={
        "share": "Share of Products",
        "price_tier": "Price Tier",
        "keyword": "Attribute",
    },
    template=PLOTLY_TEMPLATE,
    color_discrete_sequence=px.colors.qualitative.Set2,
)
fig.update_layout(yaxis_tickformat=".0%")
fig.show()

KeyError: "Columns not found: 'lightweight', 'full_coverage', 'natural_finish', 'long_wear', 'hydrating', 'spf', 'matte'"

### 4.10 Review Volume vs. Price — Social Proof at Different Price Points


In [ ]:
box_df = df.dropna(subset=["price_tier", "total_reviews"]).copy()

fig = px.box(
    box_df,
    x="price_tier",
    y="total_reviews",
    color="price_tier",
    points="all",
    log_y=True,
    title="📣 Review Volume (Social Proof) by Price Tier",
    labels={"price_tier": "Price Tier", "total_reviews": "Total Reviews (log)"},
    template=PLOTLY_TEMPLATE,
    color_discrete_sequence=px.colors.sequential.Reds_r,
)
fig.update_layout(showlegend=False)
fig.show()

### 4.11 Gen Z / Millennial Appeal Signals

Younger shoppers gravitate toward: lightweight, skin-tint, natural finish, hydrating, and dewy formulas.  
Let's score each product on these signals.


In [ ]:
genz_signals = ["lightweight", "skin_tint", "natural_finish", "hydrating", "dewy"]
df["genz_score"] = df[genz_signals].sum(axis=1)

genz_df = (
    df[df["genz_score"] > 0]
    .sort_values("genz_score", ascending=False)
    .head(20)[
        ["title", "brand", "current_price", "rating", "total_reviews", "genz_score"]
    ]
)

fig = px.scatter(
    genz_df,
    x="current_price",
    y="rating",
    size="total_reviews",
    color="genz_score",
    hover_name="title",
    color_continuous_scale="Viridis",
    title="🧬 Gen Z Appeal Score vs. Price & Rating (top 20 products)",
    labels={
        "current_price": "Price ($)",
        "rating": "Avg Rating",
        "genz_score": "Gen Z Appeal Score",
        "total_reviews": "Total Reviews",
    },
    template=PLOTLY_TEMPLATE,
)
fig.update_layout(xaxis_tickprefix="$")
fig.show()

print("\nTop 10 products by Gen Z Appeal Score:")
genz_df.head(10).to_string(index=False)

KeyError: "None of [Index(['lightweight', 'skin_tint', 'natural_finish', 'hydrating', 'dewy'], dtype='str')] are in the [columns]"

### 4.12 Badge Analysis — Best Seller & Amazon's Choice


In [ ]:
badge_summary = pd.DataFrame(
    {
        "Badge": ["Best Seller", "Amazon's Choice", "Amazon Prime", "Amazon Exclusive"],
        "Count": [
            df["is_best_seller"].sum(),
            df["is_amazon_choice"].sum(),
            df["is_amazon_prime"].sum(),
            df["is_amazon_exclusive"].sum(),
        ],
    }
)

fig = px.bar(
    badge_summary,
    x="Badge",
    y="Count",
    color="Badge",
    title="🏅 Badge Distribution Across Products",
    template=PLOTLY_TEMPLATE,
    text="Count",
)
fig.update_layout(showlegend=False)
fig.show()

### 4.13 Bought in Past Month — Demand Signals


In [ ]:
demand_df = df.dropna(subset=["bought_past_month"]).nlargest(15, "bought_past_month")[
    ["title", "brand", "bought_past_month", "price", "rating"]
]

# Shorten titles for display
demand_df["short_title"] = demand_df["title"].str[:45] + "…"

fig = px.bar(
    demand_df,
    x="bought_past_month",
    y="short_title",
    orientation="h",
    color="price",
    color_continuous_scale="RdBu_r",
    title="🔥 Top 15 Products by Units Bought in Past Month",
    labels={
        "bought_past_month": "Units Bought (Past Month)",
        "short_title": "",
        "price": "Price ($)",
    },
    template=PLOTLY_TEMPLATE,
    text="bought_past_month",
)
fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.show()

---

## 🧠 Step 6: Key Insights & MAC Recommendations

### What the data tells us:

| Insight              | Finding                                                                                                                  |
| -------------------- | ------------------------------------------------------------------------------------------------------------------------ |
| **Price sweet spot** | Majority of best-sellers cluster in the **\$10–\$30** range — MAC's premium price point faces volume pressure            |
| **Social proof gap** | Budget brands have review counts in the **10k–50k+** range; premium brands have far fewer                                |
| **Positioning**      | **Full coverage**, **long wear**, and **SPF** are most common. **Skin tint** and **dewy** are fast-growing Gen Z signals |
| **Gen Z appeal**     | Products scoring 3+ on the Gen Z scale are almost exclusively under \$30                                                 |
| **Brand dominance**  | Revlon, Maybelline, and L'Oréal dominate SKU count; MAC has limited Amazon visibility                                    |
| **Rating ceiling**   | Nearly all products rate 4.3–4.8 — ratings alone don't differentiate; review _volume_ does                               |

### 🎯 MAC Cosmetics Action Items:

1. **Introduce a sub-\$30 "Studio Lite" SKU** — capture volume-driven, younger shoppers without devaluing core MAC
2. **Revise Amazon copy** — front-load keywords like _lightweight_, _hydrating_, _natural finish_, _buildable_ in bullet points
3. **Launch a review campaign** — incentivise verified purchasers via loyalty programme; close the social-proof gap
4. **Lean into skin-tint / serum-foundation positioning** — underpenetrated by mass brands, strong Gen Z demand signal
5. **Pursue Amazon's Choice / Best Seller tags** — structured discounting or bundle strategy during key seasonal windows
